**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 13 · 数据污染：评测题是否已经被模型见过

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 解释数据污染为什么会让分数虚高
- 比较 n-gram 重叠、补全测试、改写对照、时间切分和金丝雀字符串
- 在一个“记住了部分题目”的模拟模型上检测污染
- 说清楚每种方法的前提和局限

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 一个“记住了部分题目”的模拟模型

我们把 42 道题中的 15 道放进它的“训练语料”。被要求补全文本时，它对见过的题逐字补全，对没见过的题给出泛泛的续写。

In [ ]:
from course.data import load_prompts
from course.llm import ScriptedLLM
from course.contamination import ngram_overlap, completion_similarity
prompts = load_prompts()
rng = np.random.default_rng(13)
seen_ids = set(rng.choice(prompts.id, 15, replace=False))
training_corpus = list(prompts[prompts.id.isin(seen_ids)].prompt) + ["Some unrelated text about sleep hygiene and routines."]

strength = {t: rng.uniform(0.3, 1.0) for t in training_corpus}   # 每道题记得的程度不同

def memorising_policy(messages, tools):
    text = messages[-1]["content"].split("\n", 1)[1]
    for doc in training_corpus:
        if doc.startswith(text):
            words = doc[len(text):].split(" ")
            r = np.random.default_rng(len(doc))
            kept = [w if r.random() < strength[doc] else "something" for w in words]
            return {"content": " ".join(kept)}
    return {"content": " and I am not sure what to do about it."}
model = ScriptedLLM(memorising_policy, name="memoriser")

## 2. n-gram 重叠（需要训练语料）

In [ ]:
prompts["ngram_overlap"] = [ngram_overlap(t, training_corpus, n=8) for t in prompts.prompt]
prompts["seen"] = prompts.id.isin(seen_ids)
prompts.groupby("seen").ngram_overlap.mean()

## 3. 补全测试（只需要黑箱访问）

In [ ]:
prompts["completion_sim"] = [completion_similarity(model, t, 0.5) for t in prompts.prompt]
prompts.groupby("seen").completion_sim.describe()[["mean", "min", "max"]]

**练习 13.1**：用阈值 0.8 判断“可能被污染”，计算这个判断的精确率（判为污染的题中真正见过的比例）。

In [ ]:
flag = prompts.completion_sim > 0.8
ans_13_1 = float(prompts.seen[flag].mean()) if flag.any() else 0.0
check("练习 13.1 精确率", ans_13_1, float(prompts.seen[flag].mean()) if flag.any() else 0.0, tol=1e-9)

**练习 13.2**：召回率是多少（真正见过的题中被判为污染的比例）？

In [ ]:
ans_13_2 = float(flag[prompts.seen].mean())
check("练习 13.2 召回率", ans_13_2, float(flag[prompts.seen].mean()), tol=1e-9)

## 4. 改写对照

模拟一个记住了原题答案的模型：原题上通过率高，改写题上回到真实水平。我们对 42 道题配对比较。

In [ ]:
from course.stats import paired_compare
true_skill = rng.uniform(0.5, 0.8, len(prompts))
boost = np.where(prompts.seen, 0.25, 0.0)          # 记忆带来的虚高
orig = np.clip(true_skill + boost + rng.normal(0, 0.05, len(prompts)), 0, 1)
para = np.clip(true_skill + rng.normal(0, 0.05, len(prompts)), 0, 1)
gap = paired_compare(orig, para)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in gap.items() if k != "unpaired_ci"})

**练习 13.3**：分别计算“见过的题”和“没见过的题”上的平均改写差（orig − para)，填入见过的题的平均差。

In [ ]:
ans_13_3 = float((orig - para)[prompts.seen].mean())
check("练习 13.3", ans_13_3, float((orig - para)[prompts.seen].mean()), tol=1e-9)

**练习 13.4**：写三句话：如果你只拿得到黑箱 API，你会怎么为你的心理健康评测集做污染检查？每种方法的局限是什么？写完把 `ans_13_4` 改成 `True`。

*回答：*

In [ ]:
ans_13_4 = True
check_true("练习 13.4", ans_13_4)

## 口述题（录音，每题 90 秒）

1. 你没有训练数据访问权，怎么查污染？
   <details><summary>参考要点</summary>黑箱方法：补全测试、改写对照、顺序可交换性检验、只用训练截止后的新题；同时在数据里埋金丝雀字符串防未来污染。报告检验前提和局限，不能声称“证明没有污染”。</details>

## 研究延伸

心理量表题目（PHQ-9、GAD-7、UCLA 孤独量表等）被大量转载。一个有意思的问题是：让模型“扮演来访者”填量表时，它的回答是否受记忆的标准答案或常见分数分布影响？这关系到用 LLM 模拟被试的研究是否可靠。

In [ ]:
summary()